In [1]:
%load_ext autoreload
%autoreload 2
# %matplotlib qt
from ma.qa_ma_sqi import load_sqi_df
import ipywidgets as widgets
from itertools import combinations
import numpy as np

LABEL = "sq" # ma or sq
SIGNAL_TYPE = "PPG"
CLASS_NAME = "HR"

In [ ]:
sqi_df = load_sqi_df(signal=SIGNAL_TYPE,label=LABEL)
sqi_df

## Find Label Distributions

In [ ]:
label_distributions = {}
good_sum = 0
bad_sum = 0
noisy_sum = 0
ma_sum = 0
noma_sum = 0
for parti_no in range(1, 21):
    df_filtered_parti = sqi_df[sqi_df["parti"] == str(parti_no)]
    good_labels = len(df_filtered_parti[df_filtered_parti["sq"] == 0])
    bad_labels = len(df_filtered_parti[df_filtered_parti["sq"] == 1])
    noisy_labels = len(df_filtered_parti[df_filtered_parti["sq"] == 2])
    ma_labels = len(df_filtered_parti[df_filtered_parti["ma"] == 1])
    noma_labels = len(df_filtered_parti[df_filtered_parti["ma"] == 0])
    label_distributions[parti_no] = {
        "good": good_labels,
        "bad": bad_labels,
        "noisy": noisy_labels,
        "ma": ma_labels,
        "no_ma": noma_labels,
    }

    good_sum += good_labels
    bad_sum += bad_labels
    noisy_sum += noisy_labels
    ma_sum += ma_labels
    noma_sum += noma_labels

good_percentage = good_sum/(good_sum+bad_sum+noisy_sum)
bad_percentage = bad_sum/(good_sum+bad_sum+noisy_sum)
noisy_percentage = noisy_sum/(good_sum+bad_sum+noisy_sum)
label_distributions, good_percentage, bad_percentage, noisy_percentage

## Sort Label Distributions based on Label

In [ ]:
# Sort the dictionary items by the 'good' value in descending order
sorted_items = sorted(label_distributions.items(), key=lambda x: x[1]['good'], reverse=True)

# Print the sorted dictionary
for key, value in sorted_items:
    print(f"Key: {key}, Good: {value['good']}, Bad: {value['bad']}, Noisy: {value['noisy']}, Ma: {value['ma']}, No_Ma: {value['no_ma']}")


## Linear Optimization Problem for Selecting Test Participant Set

In [ ]:
# Desired proportions
desired_good_pct = good_percentage
desired_bad_pct = bad_percentage
desired_noisy_pct = noisy_percentage

# Helper function to compute the proportions
def calculate_proportions(selected_sets):
    total_good = sum(label_distributions[set_id]['good'] for set_id in selected_sets)
    total_bad = sum(label_distributions[set_id]['bad'] for set_id in selected_sets)
    total_noisy = sum(label_distributions[set_id]['noisy'] for set_id in selected_sets)
    total = total_good + total_bad + total_noisy

    return total_good / total, total_bad / total, total_noisy / total

# Function to calculate the score for how close the proportions are to the target
def score_proportions(selected_sets):
    good_pct, bad_pct, noisy_pct = calculate_proportions(selected_sets)
    return np.sqrt((good_pct - desired_good_pct)**2 + (bad_pct - desired_bad_pct)**2 + (noisy_pct - desired_noisy_pct)**2)

# Find the best combination of 4 sets
best_sets = None
best_score = float('inf')

for sets in combinations(label_distributions.keys(), 4):
    current_score = score_proportions(sets)
    if current_score < best_score:
        best_score = current_score
        best_sets = sets

# Output the best combination and its proportions
best_good_pct, best_bad_pct, best_noisy_pct = calculate_proportions(best_sets)
print(f"Best sets: {best_sets}")
print(f"Good: {best_good_pct:.4f}, Bad: {best_bad_pct:.4f}, Noisy: {best_noisy_pct:.4f}")

--------------

## Plotting Confusion Matrixes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Compute confusion matrices
cm_train = np.array([[30640, 1582, 157], [54, 3956, 34], [10, 248, 9958]])
cm_test = np.array([[30640, 1582, 157], [54, 3956, 34], [10, 248, 9958]])
labels = ['Good', 'Bad', 'Noisy']

# Create a 1x2 grid for train and test confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot confusion matrix for the train dataset
disp_train = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=labels)
disp_train.plot(ax=axes[0], cmap=plt.cm.Blues, values_format="d")
axes[0].set_title("Train Dataset")

# Plot confusion matrix for the test dataset
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=labels)
disp_test.plot(ax=axes[1], cmap=plt.cm.Blues, values_format="d")
axes[1].set_title("Test Dataset")

# Adjust layout
plt.suptitle("Model: RUSBoosted Random Forest")
plt.tight_layout()

# Save the figure
plt.savefig("confusion_matrices_train_test.png", dpi=300)

# Display the plot
plt.show()


------------------

## Getting GP Results

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

import matplotlib.pyplot as plt
plt.rcParams['pgf.preamble'] = r'\usepackage{tikz}'
from ma.gp import GPPerfectClassifier, GPModelClassifier
import pandas as pd

In [ ]:
parti_nos = range(1,21)
SIGNAL_NAMES = ["ecg1", "ecg2", "ecg3", "ecg4"]
CLASS_NAME = "HR"
results_gp_rational_quadratic = {parti_no: {signal_name: {"errors_raw": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}, "errors_recovered": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}} for signal_name in SIGNAL_NAMES} for parti_no in parti_nos}
results_gp_combined_se = {parti_no: {signal_name: {"errors_raw": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}, "errors_recovered": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}} for signal_name in SIGNAL_NAMES} for parti_no in parti_nos}
results_gp_combined_matern = {parti_no: {signal_name: {"errors_raw": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}, "errors_recovered": {"mse": 0, "mae": 0, "rmse": 0, "med_abs_err": 0}} for signal_name in SIGNAL_NAMES} for parti_no in parti_nos}

for parti_no in parti_nos:
    try:
        gp_pc = GPPerfectClassifier(
            parti_no=parti_no, signal_names=SIGNAL_NAMES, class_name=CLASS_NAME
        )
        gp_pc.recover_signals()
        gp_pc.calculate_ma_error()
        for signal_name in SIGNAL_NAMES:
                df_results = gp_pc.get_ma_errors(signal_name=signal_name, model_name="gp_rational_quadratic")
                results_gp_rational_quadratic[parti_no][signal_name]["errors_raw"] = df_results['errors_raw']
                results_gp_rational_quadratic[parti_no][signal_name]["errors_recovered"] = df_results['errors_recovered']

                df_results = gp_pc.get_ma_errors(signal_name=signal_name, model_name="gp_combined_se")
                results_gp_combined_se[parti_no][signal_name]["errors_raw"] = df_results['errors_raw']
                results_gp_combined_se[parti_no][signal_name]["errors_recovered"] = df_results['errors_recovered']

                df_results = gp_pc.get_ma_errors(signal_name=signal_name, model_name="gp_combined_matern")
                results_gp_combined_matern[parti_no][signal_name]["errors_raw"] = df_results['errors_raw']
                results_gp_combined_matern[parti_no][signal_name]["errors_recovered"] = df_results['errors_recovered']
    except:
        print(f"Parti No: {parti_no} is failed to recovered")
